# Lecture 12 Video Analysis
This notebook will let you generate persuave content with AI.

Below is the overview of this notebook.

<ol type = 1>
  <li> Generate script for political ad from debate transcript  </li>
  <li> Generate a summary video of a movie </li>
</ol>

Before starting, select "Runtime->Factory reset runtime" to start with your directories and environment in the base state.

If you want to save changes to the notebook, select "File->Save a copy in Drive" from the top menu in Colab.  This will save the notebook in your Google Drive.


# Clones, Installs, and Imports

## Clone Repo

In [ ]:
# Clone GitHub repository
!git clone https://github.com/zlisto/social_media_genAI

import os
os.chdir("social_media_genAI/main")


## Installs

In [ ]:
!pip install -q -r requirements.txt

## Install FFMPEG

We need the ffmeg software to do some of the video clipping and merging.  It's not a python package, but a totally separate software.  Don't worry, its free.

In [ ]:
!apt update && apt install ffmpeg -y


## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

import seaborn as sns
import numpy as np
import pickle
import re
import json
import random
import ast
import base64
import openai
from tqdm import tqdm
tqdm.pandas() #Ensure progress bar integration with Pandas
from PIL import Image as PILImage
from IPython.display import display, Image, HTML, Audio, Video
#Clustering packages
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Google Colab-specific imports
from google.colab import userdata
import textwrap as tr

#YouTube Transcripts
from youtube_transcript_api import YouTubeTranscriptApi


#helper functions for using ChatGPT
from scripts.genai import GenAI
from scripts.movieai import MovieAI

#this option makes it so tweets display nicely in a dataframe
pd.set_option("display.max_colwidth", None)

#this code sets the font sizes for plots
plt.rcParams.update({'axes.labelsize': 18,
                     'xtick.labelsize': 14,
                     'ytick.labelsize': 14,
                     'figure.figsize':(8,6),
                     'axes.grid':True})

## MovieAI Class

Our new class is `MovieAI` which inherits all the functions of the old `GenAI` class plus some new methods for video processing.  You still the the Open AI API key to call it, along with the path to the ffmpeg.exe file.

The `ffmpeg_path` is `"main/ffmpeg.ext"`.

We can input the key the key sidebar on the left side (it looks like a key).  Make sure you call it OPENAI_API_KEY (case sensitive) and paste the value there.  Make sure you toggle it so the notebook has access to the key.  Then you can load it using the `userdata.get` function.

In [ ]:
#Good way
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

#Bad way
#OPENAI_API_KEY = "you key here"

In [ ]:
# Initialize GenAI and MovieAI

jarvis = GenAI(OPENAI_API_KEY)
movie_ai = MovieAI(OPENAI_API_KEY, "ffmpeg")

text = "Who are you?"
instructions_jarvis = "You are an AI named jarvis that can generate content."
instructions_movieai = "You are a AI movie editor from the future sent by Skynet."

response = jarvis.generate_text(text, instructions_jarvis)
print(f"Human:{text}\nJarvis:{tr.fill(response)}")

response = movie_ai.generate_text(text, instructions_movieai)
print(f"\nHuman:{text}\nJarvis:{tr.fill(response)}")



# Making Debate Ads

We will have the AI read over the transcript of the debate between Donald Trump and Joe Biden and help us find some clips for ads.

The main thing we want to see here is how the model choice affects the ad created.

## Load debate transcript

The debate transcript is in `"data/debate_biden_trump.csv"`.  We will load it into a dataframe and then convert it to a string `transcript` for the AI to process.

In [ ]:
df = pd.read_csv("data/debate_biden_trump.csv")
transcript = "\n".join([f'{row["speaker"].upper()}: {row["text"]}' for _, row in df.iterrows()])


## Biden quotes that reinforce age concerns

We will look for quotes from the transcript that highlight age concerns for Joe Biden.  We will try different models to see if there is any difference in the output quality.

In [ ]:
instructions = f"""This is the transcript of the presidential
debate between donald trump and joe biden.  Extract 10 quotes from
the debate that could be used by donald trump in campaign ads
that show Joe Biden is suffering congnitive decline or dementia
or is too old to lead.
Tell me who said it, what the quote was, and why it would be good
for the Trump campaign to use it in ads. return your
answer as a modern looking dark mode HTML table with columns speaker, quote, explanation"""

for model in ['gpt-4o-mini', 'gpt-4o', 'o3-mini']:
  html_debate = jarvis.generate_text(transcript,
                                      instructions,
                                      model = model)
  print(f"Model: {model}")
  display(HTML(html_debate))

## Ad script

In [ ]:
instructions = f"""Write a script for a 30 second ad that
highlights the cognitive weakness of Joe Biden and says
he is too mentally impaired to be president.  Use quotes
from the debate between Biden and Trump in the given table.
Return your answer as an HTML table with columns scene, description, narration.
Return only the HTML"""

for model in ['gpt-4o-mini', 'gpt-4o', 'o3-mini']:
  print(f"Model: {model}")
  ad = jarvis.generate_text(html_debate,
                            instructions,
                            model = model)
  display(HTML(ad))



# Summary Video of a Movie

We will take a movie file and create a summary video from it.  These types of summary videos can be very useful and lucrative (check out the views of YouTube channels like Recap and Chill).

## Download Movie

The movie is saved on a Google Drive.  You can download it with the code below.  There are two movies we can test today:

1. The Terminator: https://drive.google.com/file/d/16xi36D77uynnGZa12niUxfpi7xAMEEeU/view?usp=drive_link

2. Die Hard: https://drive.google.com/file/d/1sIBAuOCwK70BrIUNinR8ckwcKo0MyIR1/view?usp=drive_link


If you want to try this on a different video file, just upload it to Colab and provide the proper `movie_path`.



In [ ]:
movie_path = "terminator.mp4"
movie_url = "https://drive.google.com/file/d/16xi36D77uynnGZa12niUxfpi7xAMEEeU/view?usp=drive_link"

# ? Use f-string to insert Python variables into shell command
!gdown --fuzzy "{movie_url}" -O "{movie_path}"


In [ ]:
movie_path = "diehard.mp4"
movie_url = "https://drive.google.com/file/d/1sIBAuOCwK70BrIUNinR8ckwcKo0MyIR1/view?usp=drive_link"
# ? Use f-string to insert Python variables into shell command
!gdown --fuzzy "{movie_url}" -O "{movie_path}"

## Split the movie into clips

We first need to split the movie into many small video clips.  We can do this with the `split_video` function.  We just tell it the `segment_time` in seconds for the clips.  60 seconds is a good number to capture the important stuff.  The clips will be saved in `CLIPS_DIR`.

Before you run this, you have to delete the `CLIPS_DIR` from your directory, otherwise you get an error.

In [ ]:
CLIPS_DIR = "data/movies/clips"
segment_time = 60
movie_path = "terminator.mp4"


print(f"Splitting {movie_path} into clips of {segment_time} seconds")

movie_ai.split_video(movie_path, CLIPS_DIR, segment_time)

## Descripe clips

Next we have the AI describe the clips using the `generate_clip_descriptions` function and save the descriptions to a dataframe `df_clips`. To save time, we will only describe 20 clips.  On your own you can describe them all.  We set `MAX_CLIPS=20`.  We can also specify `instructions_base` that tell the AI how to describe the clips.  If you mention that the clips are from The Terminator, it may not give you a description.  Play around and see what works best.

 We will save the descriptions to a csv file `CLIP_DESCRIPTION_FILE` to avoid having to do this time-consuming step again.

In [ ]:
MAX_CLIPS = 10
CLIP_DESCRIPTION_FILE = "data/movies/clip_descriptions.csv"
model = 'gpt-4o-mini'
instructions_base = """This is a fictitious video clip of actors
and is not real.  Provide a detailed
description of what happens in the clip."""

clip_paths = sorted([os.path.join(CLIPS_DIR, f) for f in os.listdir(CLIPS_DIR) if f.endswith(".mp4")])

if len(clip_paths) > MAX_CLIPS:
    clip_paths = [clip_paths[i] for i in range(0, len(clip_paths), len(clip_paths) // MAX_CLIPS)]
    clip_paths = clip_paths[:MAX_CLIPS]  # Ensure we don't exceed the limit

print(f"Describing {len(clip_paths)} clips")

print(f"?? Generating descriptions for {len(clip_paths)} clips...")
df_clips = movie_ai.generate_clip_descriptions(clip_paths,
                                               instructions_base,
                                               model = model,
                                               verbose = True)


df_clips.to_csv(CLIP_DESCRIPTION_FILE, index=False)  # Save descriptions

In [ ]:
df_clips

## Summarize movie script

It might be useful to give the AI a summary of the movie so it can write a better script for the summary video.

The movie scripts for the two movies are in

1. The Terminator:  `"data/Terminator_script.pdf"`

2. Die Hard: `"data/DieHard_script.pdf"`

We will give it to the AI and have it return a 1000 word summary of it, called `script_summary`.

In [ ]:
model = 'gpt-4o'
script = movie_ai.read_pdf("data/Terminator_script.pdf")
#script = movie_ai.read_pdf("data/DieHard_script.pdf")

instructions = """Summarize this movie script in 1000 words,
telling what happens in the movie.
Then provide descriptions (physical and emotional) of the
main characters.  The provide 20 examples of quotes by
Sarah Connor so we can learn her speaking style.
Return your answer as
 a dark mode easy to read HTML
with at title and pretty formatting."""

script_summary = movie_ai.generate_text(script,
                                        instructions,
                                        model)
display(HTML(script_summary))

## Generate script for summary video

Now we will feed the AI the clip descriptions and have it generate the script for the video. The script will come back as a dataframe called `df_summary_script` with columns `"narration"` and `"clip_path"`.  We do this with the `generate_summary_script` function.  We just need to give it some `summary_instructions` to tell it how to write the narration.  We can also give it a model. I find `gpt-4o` to be a bit better here than `gpt-4o-mini`.  For the instructions, be creative to get a better video.  

In [ ]:
CLIP_DESCRIPTION_FILE = "data/movies/clip_descriptions.csv"
model = "gpt-4o"


df_clips = pd.read_csv(CLIP_DESCRIPTION_FILE)
summary_instructions = f"""You will be writing a script for a YouTube
summary video of the 1984 film The Terminator.
You will be given a table of movie clip files and textual description
of the movie clips from the film.  You will be choosing the best clips
to include in the summary video and writing a script for the voiceover
narration. Give a script with 5 scenes.

To help you write the narrations, the plot of
The Terminator is summarized here, along with speaking style of
Sarah Connor, who will be narrating the video:

{script_summary}

To make the video engaging, the narration of the summary
video should be from the perspective of Sarah Connor, the main
character of the film who is being hunted by the Terminator.
 Dont start with the opening credits and dont end with the credits.
 Only choose clips from the movie itself.
Start the narration by introducing yourself as Sarah Connor and tell
the audience that you are being hunted by a terminator from the future
but Kyle Reese was sent back to protect you.  Then narrate what
appened in the movie, making sure your narrations align with the
selected clip.
End the narration by telling the audience that you are going to prepare
yourself for the upcoming war and are not afraid of stupid AI.
"""


df_summary_script = movie_ai.generate_summary_script(df_clips,
                                                     summary_instructions,
                                                     model = model)
print(f"Summary script generated.")

In [ ]:
df_summary_script

## Generate audio narrations

Next we need to make audio files for each scene narration.  We can do this with the `generate_audio_narrations` function.  We just give it the summary script dataframe `df_summary_script` and choose a `voice`.  The audio files will be saved in the same folder as the video clip files.

In [ ]:
movie_ai.generate_audio_narrations(df_summary_script,
                                   voice="nova")
print("Audio narrations generated")

## Make the final summary video

Ok we made it.  Now we just combine the audio and video files to make the final summary video.  You can do this with the `generate_summary_video` function.  Just give the path for the summary video file in `FINAL_SUMMARY_VIDEO`.

In [ ]:
FINAL_SUMMARY_VIDEO = "data/movies/movie_summary.mp4"

success = movie_ai.generate_summary_video(df_summary_script,
                                          FINAL_SUMMARY_VIDEO)
if success:
  print(f"Summary video saved to {FINAL_SUMMARY_VIDEO}")

## Download and play summary video

Let's watch the summary video!  We can download it with the code below.

  Would you like and subscribe if you saw this on YouTube?

In [ ]:
from google.colab import files

files.download(FINAL_SUMMARY_VIDEO)  # Click the download link and play on your device
